In [1]:
# importy
# import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.formula.api as smf
import statsmodels.api as sm



ModuleNotFoundError: No module named 'scipy'

Zadanie 1

In [1]:
filepath = "wyniki.csv"
df = pd.read_csv(filepath)

df.head()

NameError: name 'pd' is not defined

In [200]:
# hipotezy
alpha = 0.05
hip_0 = "Nie ma istotnej różnicy w Ocenie końcowej pomiędzy uczniami a uczennicami"
hip_a = "Istnieje istotna różnica w Ocenie końcowej pomiędzy uczniami"

# Podział danych na płci
females = df[df["plec"] == "F"]["ocena_koncowa"]
males = df[df["plec"] == "M"]["ocena_koncowa"]

# scipy ttest https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html
t_stat, p_value = ttest_ind(females, males, equal_var=False)

t_stat, p_value

print(f"T-stat: {round(t_stat, 4)}")
print(f"P-val: {round(p_value, 4)}")

print(f"Ponieważ p_values wynosi ({round(p_value, 4)}) w stosunku do significance level {alpha}:")
if round(p_value, 2) > 0.05 or round(p_value, 2) < 0.05:
    print(hip_a)
else:
    print(hip_0)

T-stat: -1.9386
P-val: 0.0533
Ponieważ p_values wynosi (0.0533) w stosunku do significance level 0.05:
Nie ma istotnej różnicy w Ocenie końcowej pomiędzy uczniami a uczennicami


Zadanie 2

In [201]:
filepath = "ZyskiFirmyX.csv"
df = pd.read_csv(filepath)

hip_0 = "Nie ma widocznego trendu zysków w czasie"
hip_a = "Zysk zmienia się liniowo na przestrzeni lat"

df

,Rok,Zysk
0,1821,1645
1,1822,658
2,1823,1926
3,1824,865
4,1825,764
...,...,...
195,2016,988
196,2017,895
197,2018,-178
198,2019,543


In [202]:
px.scatter(df, "Rok","Zysk", title="Zyski osiągnięte w latach 1821-2020")

In [203]:
# Model z dostosowanymi latami y1=0
df["RokAdj"] = df["Rok"] - 1821
model = smf.ols(formula="Zysk ~ RokAdj", data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   Zysk   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.005
Method:                 Least Squares   F-statistic:                   0.09357
Date:                Sun, 22 Dec 2024   Prob (F-statistic):              0.760
Time:                        01:08:08   Log-Likelihood:                -1521.3
No. Observations:                 200   AIC:                             3047.
Df Residuals:                     198   BIC:                             3053.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   1040.2212     68.906     15.096      0.0

In [204]:
df["fitted"] = model.fittedvalues

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["Rok"], y=df["Zysk"], name="Zysk w latach 1821-2020", mode="lines+markers"))
fig.add_trace(go.Scatter(
    x=df["Rok"], y=df["fitted"], name="Fitted Regression Line"))
fig.update_layout(title="Linia regresji Zysk vs Czas", xaxis_title="Rok", yaxis_title="Zysk w mln")
fig.show()

pval = model.pvalues
coef = model.params.values

print(pval, coef)

print("Z coefficient dla lat równemu i P value ciężko zauważyć jakikolwiek trend w związku z zyskami analizowanych lat.")

Intercept    9.037946e-35
RokAdj       7.600076e-01
dtype: float64 [ 1.04022119e+03 -1.83228081e-01]
Z coefficient dla lat równemu i P value ciężko zauważyć jakikolwiek trend w związku z zyskami analizowanych lat.


Zadanie 3

In [205]:
filepath = "spendingscores.csv"
df = pd.read_csv(filepath)

df.head()

,Gender,Age,AnnualIncome,SpendingScore
0,Male,19,15,39
1,Male,21,15,81
2,Female,20,16,6
3,Female,23,16,77
4,Female,31,17,40


In [206]:
matrix = px.scatter_matrix(
    df,
    dimensions=["Age", "AnnualIncome", "SpendingScore"],
    color="Gender",
    title="Spending Score",
    labels={"Gender": "Gender"},
    height=800,
    width=800
)
matrix.show()

In [ ]:
print("Wiek jest jedynym wyznacznikiem SpendingScore. Młodsi mają tendencję do wydawania więcej.")
print("Gender i AnnualIncome nie wpływają znacząco na SpendingScore")

df["Gender"] = df["Gender"].map({
    "Male": 1,
    "Female": 0
})

x = df[["Gender", "Age", "AnnualIncome"]]
y = df["SpendingScore"]

# dodaj zmiennną niezależną
x = sm.add_constant(x)

model = sm.OLS(y, x).fit()

print("Dla modelu wejściowego:")
print("Model P Values:", model.pvalues.values)
print("Model Coef:", model.params.values)
print("Model Std Errs:", model.bse.values)

t_val = model.params[1]/model.bse[1]
p_val = stats.t.sf(abs(t_val), len(df)-2)*2
print()
print("Wyliczona P Value: ", p_val)

# eliminacja wsteczna
x_ref = x.drop(columns=["AnnualIncome"])

summary = model.summary()
print(summary)

print("Dla modelu zredefiniowanego (bez AnnualIncome)")
model_ref = sm.OLS(y, x_ref).fit()
summary_refined = model_ref.summary()
t_val = model.params[1]/model_ref.bse[1]
p_val = stats.t.sf(abs(t_val), len(df)-2)*2

print("\nWyliczona P Value: ", p_val)
print(summary_refined)

Dla modelu wejściowego:
Model P Values: [1.28775203e-22 5.67117187e-01 3.05517724e-06 9.05093933e-01]
Model Coef: [ 7.39300336e+01 -2.01323375e+00 -6.00371040e-01  7.92939510e-03]
Model Std Errs: [6.64225287 3.51182541 0.12491583 0.06641961]

Wyliczona P Value:  0.5671105551058961
                            OLS Regression Results                            
Dep. Variable:          SpendingScore   R-squared:                       0.109
Model:                            OLS   Adj. R-squared:                  0.095
Method:                 Least Squares   F-statistic:                     7.960
Date:                Sun, 22 Dec 2024   Prob (F-statistic):           4.91e-05
Time:                        01:08:08   Log-Likelihood:                -922.05
No. Observations:                 200   AIC:                             1852.
Df Residuals:                     196   BIC:                             1865.
Df Model:                           3                                         
Covaria

C:\Users\mawja\AppData\Local\Temp\ipykernel_29036\4267921867.py:19: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\mawja\AppData\Local\Temp\ipykernel_29036\4267921867.py:33: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

